# EDA espacial da base agregada (PySAL)

a análise é feita sobre os **polígonos
reais dos 96 distritos** com a pilha do PySAL (libpysal, esda, splot, spreg, mapclassify).


| Seção | O que responde |
|---|---|
| 1 | Como é o dado por distrito, e quais distritos têm amostra suficiente? |
| 2 | Como as variáveis se distribuem no mapa? (coropléticos) |
| 3 | Como definimos "vizinho"? (pesos espaciais) |
| 4 | Existe autocorrelação espacial global? Depende dos pesos? (Moran I) |
| 5 | Onde estão os aglomerados e outliers espaciais? (LISA, Getis-Ord G*) |
| 6 | Variáveis se associam no espaço? (Moran bivariado) |
| 7 | E no nível do anúncio? (Moran por número de vizinhos, resíduo do perfil) |
| 8 | O distrito é uma unidade representativa? (variância entre vs dentro) |
| 9 | Regressão exploratória com diagnóstico espacial (`spreg`) |
| 10 | Resumo |


In [1]:
import warnings, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from shapely import make_valid
from shapely.geometry import MultiPolygon

from libpysal import weights
from esda import Moran, Moran_Local, Moran_BV, G_Local, fdr
from esda.moran import Moran_Local_BV
from splot.esda import moran_scatterplot, lisa_cluster
import spreg

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 130, "axes.titleweight": "bold", "axes.titlesize": 11})

# ---- parâmetros -----------------------------------------------------------------------------
DATA = Path("../data/processed/listings_aggregated.csv")
GEO = Path("../data/external/distrito_municipal.geojson")
FIG = Path("../outputs/eda_espacial"); FIG.mkdir(parents=True, exist_ok=True)
SAVE_FIGS = True
MIN_N = 30          # mínimo de anúncios por distrito para preço/receita
PERMS = 9999
SEED = 20260503
ALPHA = 0.05
np.random.seed(SEED)

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return " ".join(s.upper().split())

def save(fig, name):
    if SAVE_FIGS:
        fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

ModuleNotFoundError: No module named 'libpysal'

## 1. Dados por distrito

Cada distrito resume os anúncios e recebe as variáveis exógenas já agregadas (crime, IPTU, POI).
O **índice de preço ajustado** é o prêmio de preço de um distrito para o *mesmo produto*: resíduo de
`log(preço) ~ tipo de quarto + capacidade + quartos` (estadias de até 3 noites, para o preço ser
comparável), tomado como mediana por distrito e convertido em %. Isso separa "distrito caro" de
"distrito com imóveis maiores".

In [ ]:
li = pd.read_csv(DATA, low_memory=False)

dist = gpd.read_file(GEO)
dist["distrito"] = dist["nm_distrito_municipal"].map(norm)
dist = dist.set_index("distrito").to_crs(31983)

def only_polygons(g):
    g = make_valid(g)
    if g.geom_type == "GeometryCollection":      # make_valid pode devolver coleção; PySAL exige polígonos
        parts = [x for x in g.geoms if x.geom_type in ("Polygon", "MultiPolygon")]
        g = parts[0] if len(parts) == 1 else MultiPolygon(
            [q for x in parts for q in (x.geoms if x.geom_type == "MultiPolygon" else [x])])
    return g
dist["geometry"] = dist.geometry.map(only_polygons)
assert set(dist.geom_type) <= {"Polygon", "MultiPolygon"}

# --- preço ajustado por perfil (nível do anúncio) ---
d = li[(li.price_num > 0) & (li.minimum_nights <= 3)].dropna(subset=["accommodates", "bedrooms"]).copy()
d["lp"] = np.log(d.price_num)
ols_perfil = smf.ols("lp ~ C(room_type) + accommodates + bedrooms", data=d).fit()
d["res"] = ols_perfil.resid
print(f"perfil explica R² = {ols_perfil.rsquared:.2f} do log do preço  (n = {len(d):,} anúncios)")

# --- tabela por distrito ---
act = li[li.estimated_revenue_l365d > 0].groupby("distrito")
tab = pd.DataFrame({"n_anuncios": li.groupby("distrito").size()})
tab["pct_ativos"] = act.size() / tab["n_anuncios"]
tab["receita_mediana"] = act["estimated_revenue_l365d"].median()
tab["ocupacao_mediana"] = act["estimated_occupancy_l365d"].median()
tab["preco_mediano"] = d.groupby("distrito")["price_num"].median()
tab["indice_preco_aj"] = (np.exp(d.groupby("distrito")["res"].median()) - 1) * 100
feat = li.groupby("distrito")[["crime_total_100k", "crime_patrimonial_100k", "crime_violento_100k",
                               "iptu_venal_m2_median", "poidens_total", "populacao_2022", "area_km2",
                               "crime_source"]].first()
tab = tab.join(feat)
tab["dens_anuncios_km2"] = tab["n_anuncios"] / tab["area_km2"]
tab["crime_km2"] = tab["crime_total_100k"] * tab["populacao_2022"] / 1e5 / tab["area_km2"]
gdf = dist[["geometry"]].join(tab)
gdf["amostra_ok"] = gdf["n_anuncios"] >= MIN_N
for c in ["receita_mediana", "preco_mediano", "ocupacao_mediana", "indice_preco_aj"]:
    gdf.loc[~gdf.amostra_ok, c] = np.nan          # medianas de poucos anúncios não entram

print(f"{len(gdf)} distritos | {gdf.amostra_ok.sum()} com ≥ {MIN_N} anúncios | "
      f"{(gdf.crime_source == 'nearest-imputed').sum()} com crime imputado")
display(gdf.drop(columns="geometry").describe().T[["count", "min", "50%", "max"]].round(1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
sns.histplot(gdf["n_anuncios"], bins=40, log_scale=True, ax=ax[0], color="#4c72b0")
ax[0].axvline(MIN_N, color="crimson", ls="--"); ax[0].set(title="Anúncios por distrito (escala log)", xlabel="n anúncios")
gdf.assign(cat=np.where(gdf.amostra_ok, f"≥ {MIN_N} anúncios (entra em preço/receita)", f"< {MIN_N} (fora)")) \
   .plot(column="cat", categorical=True, legend=True, ax=ax[1], cmap="Set2", edgecolor="white", linewidth=.3)
ax[1].set_axis_off(); ax[1].set_title("Amostra suficiente por distrito")
save(fig, "01_amostra_por_distrito"); plt.show()

## 2. Mapas coropléticos

Classes por **Natural Breaks (Jenks)** via `mapclassify`, que respeitam a assimetria dos dados melhor que
intervalos iguais. Hachura = distrito fora da amostra mínima. Variáveis muito assimétricas entram em log.

In [ ]:
gdf["log_dens_anuncios"] = np.log1p(gdf["dens_anuncios_km2"])
gdf["log_receita"] = np.log(gdf["receita_mediana"])
gdf["log_crime_km2"] = np.log1p(gdf["crime_km2"])
gdf["log_poi"] = np.log1p(gdf["poidens_total"])

PANELS = [
    ("log_dens_anuncios", "Densidade de anúncios (log por km²)", "YlGnBu"),
    ("indice_preco_aj", "Preço ajustado por perfil (% vs esperado)", "RdBu_r"),
    ("log_receita", "Receita estimada mediana 12m (log)", "YlGnBu"),
    ("pct_ativos", "Parcela de anúncios ativos", "YlGnBu"),
    ("crime_total_100k", "Crime total por 100 mil habitantes", "YlOrRd"),
    ("log_crime_km2", "Crime total por km² (log)", "YlOrRd"),
    ("iptu_venal_m2_median", "IPTU: valor venal mediano por m²", "PuRd"),
    ("log_poi", "POIs (OSM) por km² (log)", "PuBu"),
]
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
for ax, (col, title, cmap) in zip(axes.ravel(), PANELS):
    gdf.plot(column=col, scheme="NaturalBreaks", k=5, cmap=cmap, legend=True, ax=ax, edgecolor="white",
             linewidth=.25, legend_kwds={"fontsize": 7, "loc": "lower left", "fmt": "{:.1f}"},
             missing_kwds={"color": "#e6e6e6", "hatch": "///", "edgecolor": "#aaaaaa"})
    ax.set_title(title, fontsize=10); ax.set_axis_off()
fig.suptitle("Variáveis por distrito (cinza hachurado = amostra < %d anúncios)" % MIN_N, fontsize=13)
fig.tight_layout(); save(fig, "02_mapas_coropleticos"); plt.show()

## 3. Pesos espaciais

A definição de "vizinho" é uma escolha de modelagem. Usamos **Queen** (distritos que se tocam em aresta ou
vértice) como padrão e comparamos com **Rook**, **KNN** (k = 5 e 8, por centroide) na seção 4. Pesos
padronizados por linha (`'r'`): a defasagem espacial de um distrito é a média dos vizinhos.

Nas variáveis que existem só para parte dos distritos (preço, receita), os pesos são refeitos sobre o
subconjunto e **ilhas** (distritos sem nenhum vizinho no subconjunto) são removidas, porque o Moran não é
definido para elas.

In [ ]:
def queen(frame):
    w = weights.Queen.from_dataframe(frame, use_index=True); w.transform = "r"; return w

def connected(frame):
    """Remove ilhas até o subconjunto ficar conexo por contiguidade Queen."""
    f = frame.copy()
    while True:
        w = weights.Queen.from_dataframe(f, use_index=True)
        if not w.islands: return f
        print("  ilhas removidas:", w.islands); f = f.drop(index=w.islands)

W_all = queen(gdf)
sub = connected(gdf[gdf.amostra_ok]); W_sub = queen(sub)
print(f"Queen (96 distritos): ilhas = {W_all.islands}, vizinhos médios = {W_all.mean_neighbors:.2f}, "
      f"min/max = {W_all.min_neighbors}/{W_all.max_neighbors}")
print(f"Queen (subconjunto, {len(sub)} distritos): vizinhos médios = {W_sub.mean_neighbors:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, (frame, w, ttl) in zip(ax, [(gdf, W_all, "96 distritos"), (sub, W_sub, f"{len(sub)} distritos (amostra ≥ {MIN_N})")]):
    frame.plot(ax=a, color="#f2f2f2", edgecolor="#bbbbbb", linewidth=.4)
    cen = frame.geometry.centroid; pos = dict(zip(frame.index, zip(cen.x, cen.y)))
    for i, nb in w.neighbors.items():
        for j in nb:
            if str(i) < str(j): a.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]], color="#4c72b0", lw=.6)
    a.scatter(cen.x, cen.y, s=8, color="#c44e52", zorder=3); a.set_axis_off(); a.set_title(f"Vizinhança Queen: {ttl}")
save(fig, "03_grafo_vizinhanca"); plt.show()

## 4. Autocorrelação espacial global (Moran I)

`I > 0`: distritos vizinhos se parecem (aglomeração); `I ≈ 0`: sem padrão espacial; `I < 0`: vizinhos se
opõem. A significância vem de **9.999 permutações** (`p_sim`), sem supor normalidade. A tabela mostra
também se a conclusão **depende dos pesos** escolhidos.

In [ ]:
ALL_VARS = {"log_dens_anuncios": "Densidade de anúncios (log)", "crime_total_100k": "Crime por 100 mil",
            "crime_patrimonial_100k": "Crime patrimonial por 100 mil", "log_crime_km2": "Crime por km² (log)",
            "iptu_venal_m2_median": "IPTU venal/m²", "log_poi": "POIs por km² (log)"}
SUB_VARS = {"preco_mediano": "Preço mediano", "indice_preco_aj": "Preço ajustado por perfil",
            "log_receita": "Receita estimada mediana (log)", "ocupacao_mediana": "Ocupação estimada",
            "pct_ativos": "Parcela de ativos"}

labels = {**ALL_VARS, **SUB_VARS}

def moran_table(frame, w, variables, perms=PERMS):
    rows = []
    for v, lab in variables.items():
        m = Moran(frame[v].values, w, permutations=perms)
        rows.append({"variavel": lab, "n": len(frame), "I": m.I, "E[I]": m.EI, "z (perm)": m.z_sim, "p_sim": m.p_sim})
    return pd.DataFrame(rows)

mt = pd.concat([moran_table(gdf, W_all, ALL_VARS), moran_table(sub, W_sub, SUB_VARS)], ignore_index=True)
display(mt.round(3))

In [ ]:
# sensibilidade da conclusão ao critério de vizinhança
def variants(frame):
    return {"Queen": weights.Queen.from_dataframe(frame, use_index=True),
            "Rook": weights.Rook.from_dataframe(frame, use_index=True),
            "KNN 5": weights.KNN.from_dataframe(frame, k=5),
            "KNN 8": weights.KNN.from_dataframe(frame, k=8)}

rows = []
for frame, variables in ((gdf, ALL_VARS), (sub, SUB_VARS)):
    for name, w in variants(frame).items():
        w.transform = "r"
        for v, lab in variables.items():
            m = Moran(frame[v].values, w, permutations=999)
            rows.append({"variavel": lab, "pesos": name, "I": m.I, "p_sim": m.p_sim})
sens = pd.DataFrame(rows).pivot(index="variavel", columns="pesos", values="I")[["Queen", "Rook", "KNN 5", "KNN 8"]]
display(sens.round(2))
print("Todos os I positivos e p_sim < 0,05 em todas as combinações:",
      bool((pd.DataFrame(rows).I > 0).all() and (pd.DataFrame(rows).p_sim < ALPHA).all()))

**Cuidado com o crime imputado.** 20 distritos não têm DP e herdam o valor do vizinho mais próximo. Isso cria
suavidade espacial *por construção* e pode inflar o Moran do crime. Para medir o efeito, repete-se o teste só
com os distritos de crime **direto** (`crime_source == 'dp-direct'`). O **IPTU** entra como controle: ele não
tem imputação, então, se também cair, a queda vem de remover distritos (os imputados incluem vários centrais)
e não da imputação. Só o que cai **além** do controle é atribuível à imputação.

In [ ]:
dire = connected(gdf[gdf.crime_source == "dp-direct"]); W_dire = queen(dire)
rows = []
for v in ("crime_total_100k", "log_crime_km2", "iptu_venal_m2_median"):
    a = Moran(gdf[v].values, W_all, permutations=PERMS); b = Moran(dire[v].values, W_dire, permutations=PERMS)
    rows.append({"variavel": labels[v] if v in labels else v, "I (96 distritos)": a.I, "I (só crime direto)": b.I, "n direto": len(dire), "p_sim direto": b.p_sim})
imp_ef = pd.DataFrame(rows).set_index("variavel"); display(imp_ef.round(3))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
items = [("crime_total_100k", gdf, W_all), ("log_crime_km2", gdf, W_all), ("iptu_venal_m2_median", gdf, W_all),
         ("log_poi", gdf, W_all), ("indice_preco_aj", sub, W_sub), ("log_receita", sub, W_sub),
         ("ocupacao_mediana", sub, W_sub), ("pct_ativos", sub, W_sub)]
for ax, (v, frame, w) in zip(axes.ravel(), items):
    m = Moran(frame[v].values, w, permutations=PERMS)
    moran_scatterplot(m, ax=ax, p=ALPHA)
    ax.set_title(f"{labels[v]}\nI = {m.I:.2f}  (p_sim = {m.p_sim:.4f})", fontsize=9)
    ax.set_xlabel("valor padronizado"); ax.set_ylabel("média dos vizinhos")
fig.tight_layout(); save(fig, "04_moran_scatterplots"); plt.show()

## 5. Aglomerados locais (LISA) e pontos quentes (Getis-Ord G*)

O **Moran local** classifica cada distrito pela relação com a vizinhança:
**HH** (alto cercado de alto), **LL** (baixo cercado de baixo), **HL** e **LH** (outliers espaciais).
Só aparecem os distritos significativos a 5% por permutação.

Como são 96 (ou ~70) testes simultâneos, mostramos também quantos aglomerados sobrevivem à correção de
**taxa de falsas descobertas (FDR)**, que é mais rigorosa: os aglomerados "FDR" são os mais confiáveis.

In [ ]:
def lisa_run(frame, w, v):
    lm = Moran_Local(frame[v].values, w, permutations=PERMS, seed=SEED)
    p_fdr = fdr(lm.p_sim, 0.10)
    q = pd.Series(np.where(lm.p_sim < ALPHA, lm.q, 0), index=frame.index).map({0: "ns", 1: "HH", 2: "LH", 3: "LL", 4: "HL"})
    q_fdr = pd.Series(np.where(lm.p_sim <= p_fdr, lm.q, 0), index=frame.index).map({0: "ns", 1: "HH", 2: "LH", 3: "LL", 4: "HL"})
    return lm, q, q_fdr

LISA_VARS = [("log_receita", sub, W_sub), ("indice_preco_aj", sub, W_sub), ("crime_total_100k", gdf, W_all),
             ("iptu_venal_m2_median", gdf, W_all), ("log_poi", gdf, W_all), ("pct_ativos", sub, W_sub)]
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
lisa_res = {}
for ax, (v, frame, w) in zip(axes.ravel(), LISA_VARS):
    lm, q, q_fdr = lisa_run(frame, w, v)
    lisa_res[v] = (frame.index, lm, q, q_fdr)
    gdf.plot(ax=ax, color="#f4f4f4", edgecolor="#d9d9d9", linewidth=.3)      # fundo: os 96 distritos
    lisa_cluster(lm, frame, p=ALPHA, ax=ax, legend=True, legend_kwds={"loc": "lower left", "fontsize": 8})
    cnt = q.value_counts().drop("ns", errors="ignore").to_dict()
    ax.set_title(f"{labels[v]}\n{cnt}  |  sobrevivem a FDR 10%: {int((q_fdr != 'ns').sum())}", fontsize=9)
    ax.set_axis_off()
fig.suptitle("Moran local (LISA), p < 0,05 por permutação", fontsize=13)
fig.tight_layout(); save(fig, "05_lisa_clusters"); plt.show()

rows = []
for v, (idx, lm, q, q_fdr) in lisa_res.items():
    for k in ("HH", "LL", "HL", "LH"):
        rows.append({"variavel": labels[v], "tipo": k, "distritos": ", ".join(i.title() for i in idx[q == k])})
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(pd.DataFrame(rows).query("distritos != ''").set_index(["variavel", "tipo"]))

In [ ]:
# Getis-Ord G*: pontos quentes e frios (z local contra a média global)
def gstar(frame, w, v):
    g = G_Local(frame[v].values, w, star=True, permutations=PERMS, seed=SEED)
    z, p = g.Zs, g.p_sim
    cls = np.where((p < ALPHA) & (z > 0), "ponto quente", np.where((p < ALPHA) & (z < 0), "ponto frio", "ns"))
    return pd.Series(cls, index=frame.index)

W_all_star = weights.Queen.from_dataframe(gdf, use_index=True); W_all_star.transform = "r"
W_sub_star = weights.Queen.from_dataframe(sub, use_index=True); W_sub_star.transform = "r"
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
pal = {"ponto quente": "#d6604d", "ponto frio": "#4393c3", "ns": "#eeeeee"}
for ax, (v, frame, w) in zip(axes, [("log_receita", sub, W_sub_star), ("crime_total_100k", gdf, W_all_star), ("iptu_venal_m2_median", gdf, W_all_star)]):
    cls = gstar(frame, w, v)
    gdf.plot(ax=ax, color="#f4f4f4", edgecolor="#d9d9d9", linewidth=.3)
    frame.plot(ax=ax, color=cls.map(pal), edgecolor="white", linewidth=.3)
    ax.set_axis_off(); ax.set_title(f"{labels[v]}\n{cls.value_counts().drop('ns', errors='ignore').to_dict()}", fontsize=10)
    for tipo in ("ponto quente", "ponto frio"):
        print(f"{labels[v]} | {tipo}: " + ", ".join(i.title() for i in cls.index[cls == tipo]))
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=c, label=l) for l, c in pal.items()], loc="lower center", ncol=3)
fig.suptitle("Getis-Ord G* (p < 0,05)", fontsize=13); fig.tight_layout(rect=(0, .04, 1, 1))
save(fig, "06_getis_ord"); plt.show()

## 6. Associação espacial entre variáveis (Moran bivariado)

O Moran I **bivariado** mede se o valor de `X` num distrito se relaciona com o valor de `Y` nos
**vizinhos**. É diferente da correlação comum: captura se a *vizinhança* de um distrito de alta receita tende
a ter muito crime, IPTU alto etc. Os valores de `X` e `Y` são padronizados internamente.

In [ ]:
PAIRS = [("crime_total_100k", "log_receita"), ("log_crime_km2", "log_receita"), ("iptu_venal_m2_median", "log_receita"),
         ("log_poi", "log_receita"), ("crime_total_100k", "indice_preco_aj"), ("iptu_venal_m2_median", "indice_preco_aj"),
         ("log_poi", "indice_preco_aj"), ("crime_total_100k", "iptu_venal_m2_median")]
rows = []
for a, b in PAIRS:
    frame, w = (sub, W_sub) if b in ("log_receita", "indice_preco_aj") else (gdf, W_all)
    bv = Moran_BV(frame[a].values, frame[b].values, w, permutations=PERMS)
    rows.append({"X (distrito)": labels[a], "Y (vizinhos)": labels[b], "n": len(frame), "I_bivariado": bv.I, "p_sim": bv.p_sim})
bvt = pd.DataFrame(rows); display(bvt.round(3))

In [ ]:
# mapa LISA bivariado: crime por km² do distrito x receita dos vizinhos
a, b = "log_crime_km2", "log_receita"
lbv = Moran_Local_BV(sub[a].values, sub[b].values, W_sub, permutations=PERMS, seed=SEED)
fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="#f4f4f4", edgecolor="#d9d9d9", linewidth=.3)
lisa_cluster(lbv, sub, p=ALPHA, ax=ax, legend=True, legend_kwds={"loc": "lower left", "fontsize": 8})
ax.set_title("LISA bivariado: crime/km² no distrito × receita nos vizinhos (p < 0,05)"); ax.set_axis_off()
save(fig, "07_lisa_bivariado"); plt.show()

## 7. Nível do anúncio: onde a localização importa além do produto

Aqui a unidade é o anúncio (≈ 34 mil pontos). Pesos **KNN** por coordenada (em metros, EPSG:31983). Medimos
o Moran do log do preço e do **resíduo do perfil** (o que sobra depois de descontar tipo, capacidade e
quartos), para ver se a localização explica preço *além* dos atributos. O terceiro valor remove também a
média do distrito: se ainda houver autocorrelação, existe estrutura espacial **dentro** do distrito que a
unidade distrito não captura.

In [ ]:
pg = gpd.GeoDataFrame(d[["lp", "res", "distrito"]].copy(), geometry=gpd.points_from_xy(d.longitude, d.latitude), crs=4326).to_crs(31983)
pg["res_dentro"] = pg["res"] - pg.groupby("distrito")["res"].transform("mean")
xy = np.c_[pg.geometry.x, pg.geometry.y]
rows = []
for k in (5, 10, 25, 50, 100):
    wp = weights.KNN.from_array(xy, k=k); wp.transform = "r"
    rows.append({"k vizinhos": k,
                 "log preço": Moran(pg["lp"].values, wp, permutations=0).I,
                 "resíduo do perfil": Moran(pg["res"].values, wp, permutations=0).I,
                 "resíduo − média do distrito": Moran(pg["res_dentro"].values, wp, permutations=0).I})
corr = pd.DataFrame(rows).set_index("k vizinhos"); display(corr.round(3))
fig, ax = plt.subplots(figsize=(7.5, 4.5)); corr.plot(marker="o", ax=ax)
ax.set(title="Autocorrelação espacial dos anúncios por escala (KNN)", ylabel="Moran I"); ax.set_ylim(bottom=0)
save(fig, "08_moran_por_escala_anuncios"); plt.show()

wp = weights.KNN.from_array(xy, k=10); wp.transform = "r"
np.random.seed(SEED)
chk = Moran(pg["res"].values, wp, permutations=199)
print(f"Teste de permutação (k = 10, resíduo do perfil): I = {chk.I:.3f}, p_sim = {chk.p_sim:.3f}")

## 8. O distrito é uma unidade representativa?

Quanto da variância **entre anúncios** é explicada por saber só o distrito? É o `η²` (razão de correlação),
aqui na versão ajustada pelo número de grupos. Valores baixos significam que a maior parte da variação
está *dentro* dos distritos, e que a unidade distrito esconde muita heterogeneidade. Compara-se com as 5
regiões (Centro, Leste, Norte, Oeste, Sul).

In [ ]:
def eta2_adj(y, g):
    y = pd.Series(np.asarray(y, float)); g = pd.Series(np.asarray(g)); ok = y.notna() & g.notna(); y, g = y[ok], g[ok]
    n, k = len(y), g.nunique()
    sst = ((y - y.mean()) ** 2).sum(); ssw = ((y - y.groupby(g).transform("mean")) ** 2).sum()
    return 1 - (ssw / (n - k)) / (sst / (n - 1))

reg_map = li.drop_duplicates("distrito").set_index("distrito")["regiao5"]
d["regiao5"] = d["distrito"].map(reg_map)
ac = li[li.estimated_revenue_l365d > 0].copy(); ac["lr"] = np.log(ac.estimated_revenue_l365d); ac["regiao5"] = ac["distrito"].map(reg_map)
rows = [("log preço (estadias curtas)", d["lp"], d["distrito"], d["regiao5"]),
        ("resíduo do perfil", d["res"], d["distrito"], d["regiao5"]),
        ("log receita estimada (ativos)", ac["lr"], ac["distrito"], ac["regiao5"])]
eta = pd.DataFrame([{"variável": n, "η² ajustado, distrito": eta2_adj(y, g), "η² ajustado, região (5)": eta2_adj(y, r)} for n, y, g, r in rows]).set_index("variável")
display(eta.round(3))

## 9. Regressão exploratória com diagnóstico espacial (`spreg`)

Não é um modelo final: serve para perguntar se **o que as variáveis exógenas não explicam** ainda tem
estrutura espacial. Estima-se por MQO a receita mediana (log) no subconjunto com amostra suficiente, com
variáveis padronizadas, e aplicam-se os testes de Moran nos resíduos e os **multiplicadores de Lagrange**
(LM) para dependência no lag ou no erro. Se um deles for significativo, ajusta-se o modelo espacial
correspondente por máxima verossimilhança.

In [ ]:
fr = sub.copy()
fr["log_iptu"] = np.log(fr["iptu_venal_m2_median"])
cols = ["crime_total_100k", "log_iptu", "log_poi"]
Xz = ((fr[cols] - fr[cols].mean()) / fr[cols].std()).values
y = fr[["log_receita"]].values
ols = spreg.OLS(y, Xz, w=W_sub, spat_diag=True, moran=True, name_y="log receita", name_x=["crime_100k", "log_iptu", "log_poi"], name_w="Queen")
diag = pd.DataFrame({"estatística": [ols.moran_res[1], ols.lm_lag[0], ols.lm_error[0], ols.rlm_lag[0], ols.rlm_error[0]],
                     "p": [ols.moran_res[2], ols.lm_lag[1], ols.lm_error[1], ols.rlm_lag[1], ols.rlm_error[1]]},
                    index=["Moran I dos resíduos (z)", "LM lag", "LM erro", "LM lag robusto", "LM erro robusto"])
print(f"MQO: R² = {ols.r2:.2f}, n = {ols.n}")
coef = pd.DataFrame({"coef (padronizado)": ols.betas.ravel(), "p": [t[1] for t in ols.t_stat]}, index=["const"] + ["crime_100k", "log_iptu", "log_poi"])
display(coef.round(3)); display(diag.round(4))

In [ ]:
lag_sig = ols.lm_lag[1] < ALPHA; err_sig = ols.lm_error[1] < ALPHA
if lag_sig or err_sig:
    use_lag = (ols.rlm_lag[1] < ols.rlm_error[1]) if (lag_sig and err_sig) else lag_sig
    mod = (spreg.ML_Lag if use_lag else spreg.ML_Error)(y, Xz, w=W_sub, name_y="log receita",
                                                         name_x=["crime_100k", "log_iptu", "log_poi"], name_w="Queen")
    nome = "defasagem espacial (ML_Lag)" if use_lag else "erro espacial (ML_Error)"
    par = {"rho": getattr(mod, "rho", None), "lambda": getattr(mod, "lam", None)}
    print(f"Modelo escolhido pelos LM: {nome}")
    print("  parâmetro espacial:", {k: round(float(v), 3) for k, v in par.items() if v is not None},
          "| AIC OLS = %.1f → AIC espacial = %.1f" % (ols.aic, mod.aic))
    print("  coeficientes padronizados:", dict(zip(["const", "crime_100k", "log_iptu", "log_poi"], np.round(np.asarray(mod.betas[:4]).ravel(), 3))))
else:
    print("Nenhum LM significativo: sem evidência de dependência espacial residual neste modelo.")

## 10. Resumo

Os números abaixo são calculados a partir das células anteriores (não digitados).

In [ ]:
iA = mt.set_index("variavel")["I"]
print("AUTOCORRELAÇÃO GLOBAL (Queen)")
print("  maior  :", iA.idxmax(), f"(I = {iA.max():.2f})")
print("  menor  :", iA.idxmin(), f"(I = {iA.min():.2f})")
print("  todos significativos a 5%:", bool((mt['p_sim'] < ALPHA).all()))
print("  crime por 100 mil vs por km²: I = %.2f vs %.2f" % (iA["Crime por 100 mil"], iA["Crime por km² (log)"]))
print("\nLISA: distritos com aglomerado significativo (5%) / sobrevivem a FDR 10%")
for v, (idx, lm, q, q_fdr) in lisa_res.items():
    print(f"  {labels[v]:34s} {int((q != 'ns').sum()):3d} / {int((q_fdr != 'ns').sum()):3d}")
print("\nMORAN BIVARIADO significativo (p < 0,05):")
for _, r in bvt[bvt.p_sim < ALPHA].iterrows(): print(f"  {r['X (distrito)']} → {r['Y (vizinhos)']}: I = {r['I_bivariado']:.2f}")
print("  sem associação espacial:", "; ".join(f"{r['X (distrito)']} → {r['Y (vizinhos)']}" for _, r in bvt[bvt.p_sim >= ALPHA].iterrows()) or "-")
print("\nANÚNCIOS (k = 10): Moran do preço %.2f | resíduo do perfil %.2f | resíduo sem média do distrito %.2f"
      % tuple(corr.loc[10, ["log preço", "resíduo do perfil", "resíduo − média do distrito"]]))
print("\nREPRESENTATIVIDADE (η² ajustado, distrito):", {k: round(v, 2) for k, v in eta["η² ajustado, distrito"].items()})